<a href="https://colab.research.google.com/github/CelestineAnyaegbunamUgwu/-MaternalCare-AI/blob/main/Intelligent_Software_Defect_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q xgboost lightgbm catboost imbalanced-learn shap reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, classification_report, roc_curve
)

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/MyDrive/datasets/baseline.csv")
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/datasets/baseline.csv'

In [ ]:
# Remove SHA (not useful)
df = df.drop(columns=["SHA"])

# Convert target
df["defect"] = df["defect"].astype(int)

# Fill missing values
df.fillna(df.median(numeric_only=True), inplace=True)

df.head()

In [ ]:
X = df.drop("defect", axis=1)
y = df["defect"]

print("Class distribution:\n", y.value_counts())

In [ ]:
selector = SelectKBest(mutual_info_classif, k=15)

X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print(selected_features)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

In [ ]:
smote = SMOTE(random_state=42)

X_bal, y_bal = smote.fit_resample(X_scaled, y)

print("After SMOTE:\n", pd.Series(y_bal).value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_bal
)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=300),
    "XGBoost": XGBClassifier(n_estimators=300, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(n_estimators=300),
    "CatBoost": CatBoostClassifier(verbose=0)
}

In [ ]:
results = []

for name, model in models.items():
    print("Training:", name)

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, prob),
        "MCC": matthews_corrcoef(y_test, pred)
    })

results_df = pd.DataFrame(results)
results_df.sort_values("ROC_AUC", ascending=False)

In [ ]:
stack_model = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=200)),
        ("xgb", XGBClassifier(eval_metric='logloss')),
        ("lgbm", LGBMClassifier())
    ],
    final_estimator=LogisticRegression()
)

stack_model.fit(X_train, y_train)

stack_pred = stack_model.predict(X_test)
stack_prob = stack_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, stack_pred))

In [ ]:
cm = confusion_matrix(y_test, stack_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Stacking Model")
cm_path = "confusion_matrix.pdf"
plt.title("Confusion Matrix - Stacking Model")
plt.savefig(cm_path)
plt.close()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, stack_prob)

plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.title("ROC Curve")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.show()


In [ ]:
rf = RandomForestClassifier(n_estimators=300)
rf.fit(X_train, y_train)

importance = pd.DataFrame({
    "Feature": selected_features,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

importance.head(10)

In [ ]:
import shap

xgb = XGBClassifier(eval_metric='logloss')
xgb.fit(X_train, y_train)

explainer = shap.Explainer(xgb)
shap_values = explainer(X_test[:300])

shap.plots.beeswarm(shap_values)

In [ ]:
# =========================
# 1. INSTALL DEPENDENCIES
# =========================
!pip install -q xgboost lightgbm catboost imbalanced-learn shap reportlab

# =========================
# 2. IMPORT LIBRARIES
# =========================
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, classification_report, roc_curve
)

import shap
import warnings
warnings.filterwarnings("ignore")

# =========================
# 3. CREATE OUTPUT FOLDER
# =========================
save_dir = "/content/media"
os.makedirs(save_dir, exist_ok=True)

# =========================
# 4. LOAD DATASET (FIXED)
# =========================
df = pd.read_csv("/content/baseline.csv")

print("Dataset shape:", df.shape)
df.head()
# CHANGE PATH IF NEEDED
df = pd.read_csv("/content/baseline.csv")

print("Dataset shape:", df.shape)
df.head()

# =========================
# 5. PREPROCESSING
# =========================
df = df.drop(columns=["SHA"], errors="ignore")
df["defect"] = df["defect"].astype(int)
df.fillna(df.median(numeric_only=True), inplace=True)

X = df.drop("defect", axis=1)
y = df["defect"]

print("Class distribution:\n", y.value_counts())

# =========================
# 6. FEATURE SELECTION
# =========================
selector = SelectKBest(mutual_info_classif, k=15)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected Features:\n", selected_features)

# =========================
# 7. SCALING + SMOTE
# =========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X_scaled, y)

print("After SMOTE:\n", pd.Series(y_bal).value_counts())

# =========================
# 8. TRAIN-TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_bal
)

# =========================
# 9. MODELS
# =========================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=300),
    "XGBoost": XGBClassifier(n_estimators=300, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(n_estimators=300),
    "CatBoost": CatBoostClassifier(verbose=0)
}

results = []

# =========================
# 10. TRAIN + EVALUATE
# =========================
for name, model in models.items():
    print("Training:", name)

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, prob),
        "MCC": matthews_corrcoef(y_test, pred)
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values("ROC_AUC", ascending=False))

# =========================
# 11. STACKING MODEL
# =========================
stack_model = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=200)),
        ("xgb", XGBClassifier(eval_metric='logloss')),
        ("lgbm", LGBMClassifier())
    ],
    final_estimator=LogisticRegression()
)

stack_model.fit(X_train, y_train)

stack_pred = stack_model.predict(X_test)
stack_prob = stack_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, stack_pred))

cm = confusion_matrix(y_test, stack_pred)

# =========================
# 12. CONFUSION MATRIX
# =========================
cm_path = f"{save_dir}/confusion_matrix.png"

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Stacking Model")
plt.savefig(cm_path, bbox_inches="tight")
plt.show()
plt.close()

# =========================
# 13. ROC CURVE
# =========================
fpr, tpr, _ = roc_curve(y_test, stack_prob)

roc_path = f"{save_dir}/roc_curve.png"

plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.title("ROC Curve")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.savefig(roc_path, bbox_inches="tight")
plt.show()
plt.close()

# =========================
# 14. FEATURE IMPORTANCE
# =========================
rf = RandomForestClassifier(n_estimators=300)
rf.fit(X_train, y_train)

importance = pd.DataFrame({
    "Feature": selected_features,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

imp_path = f"{save_dir}/feature_importance.png"

plt.figure(figsize=(8,5))
sns.barplot(data=importance.head(10), x="Importance", y="Feature")
plt.title("Top Feature Importance")
plt.savefig(imp_path, bbox_inches="tight")
plt.show()
plt.close()

# =========================
# 15. SHAP EXPLANATION
# =========================
xgb = XGBClassifier(eval_metric='logloss')
xgb.fit(X_train, y_train)

explainer = shap.Explainer(xgb)
shap_values = explainer(X_test[:200])

shap.summary_plot(shap_values, show=True)

# =========================
# 16. PDF REPORT GENERATION
# =========================
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table
from reportlab.lib.styles import getSampleStyleSheet

pdf_path = f"{save_dir}/ML_Defect_Prediction_Report.pdf"
pdf = SimpleDocTemplate(pdf_path)
styles = getSampleStyleSheet()

content = []

content.append(Paragraph("Software Defect Prediction Report", styles["Title"]))
content.append(Spacer(1, 12))

content.append(Paragraph(f"Dataset Shape: {df.shape}", styles["Normal"]))
content.append(Spacer(1, 12))

# Model table
table_data = [results_df.columns.to_list()] + results_df.round(4).values.tolist()
content.append(Paragraph("Model Comparison Results", styles["Heading2"]))
content.append(Table(table_data))
content.append(Spacer(1, 12))

# Images
content.append(Paragraph("Confusion Matrix", styles["Heading2"]))
content.append(Image(cm_path, width=400, height=300))

content.append(Paragraph("ROC Curve", styles["Heading2"]))
content.append(Image(roc_path, width=400, height=300))

content.append(Paragraph("Feature Importance", styles["Heading2"]))
content.append(Image(imp_path, width=400, height=300))

pdf.build(content)

print("PDF saved at:", pdf_path)

# =========================
# 17. DOWNLOAD PDF
# =========================
from google.colab import files
files.download(pdf_path)